# SAM 3D Objects — 트라이온 결과 2D → 3D 변환 성능 검증 (간략 스파이크)

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

가상 시착(TryOn)이 만든 "옷 입은 사람" 2D 이미지 + 피사체 마스크를 넣어
[`facebook/sam-3d-objects`](https://huggingface.co/facebook/sam-3d-objects) 로 3D(Gaussian
splat)를 뽑고, **360° 턴테이블로 렌더해 노트북 화면에 띄우는 것까지**만 확인한다.

3D 활용 방향(360° 뷰어냐 상품 의류 3D 카탈로그냐)을 정하기 **전에** 성능이 나오는지부터 본다.

- 모델: `facebook/sam-3d-objects` — **gated repo**(접근 요청 승인 필요)
- 코드: [`facebookresearch/sam-3d-objects`](https://github.com/facebookresearch/sam-3d-objects)
  — transformers 에 들어 있지 않다. **리포를 클론해서 자체 conda 환경을 만들어야 한다.**
- 실행 환경: 로컬 노트북, RTX 4070 Laptop / VRAM 8GB
- 범위: 로컬 재구성 + 렌더 확인만. 파인튜닝·서비스 배관은 범위 밖

### 이 노트북이 답하려는 질문

1. 8GB VRAM 안에서 재구성이 **돌아가는가** (§1, §3) — upstream 은 VRAM 요구치를 공개하지 않았다
2. 1장당 **소요 시간**이 얼마인가 (§3)
3. 트라이온 출력(백색 배경 스튜디오 샷)의 **마스크를 자동으로 뽑을 수 있는가** (§2)
4. 턴테이블 결과가 **"돌려보는 확인용"으로 쓸 만한 품질인가** (§4)
5. **의류만 크롭**해서 넣는 쪽이 전신 통짜보다 나은가 (§6) — 논문 F절(단일 객체 크롭 시 상대
   해상도 상승)이 예측하는 방향을 실제로 확인한다

### 미리 알고 갈 것 (코드에서 확인된 사실)

- 이 래퍼의 출력은 **Gaussian splat 뿐이다.** `notebook/inference.py` 의 `Inference.__call__`
  이 `with_mesh_postprocess=False`, `with_texture_baking=False` 로 파이프라인을 호출한다 —
  즉 `.glb`/`.obj` 메시는 이 경로로 안 나온다. 필요하면 그 플래그를 켜는 별도 실험이 필요하다.
- **신체와 의류는 한 덩어리로 나온다.** 레이어 분리가 없으므로 "옷만 교체/편집"은 이 모델로
  불가능하다. 정적 360° 뷰어 또는 의류 단독 3D 에셋 두 방향만 후보다.

## 0. 환경 준비 — 이 노트북을 실행하기 전에 (셀이 아니라 터미널에서)

`sam-3d-objects` 는 pip 한 줄로 안 깔린다. **리포 클론 + 전용 conda 환경 + gated 체크포인트
다운로드**가 선행 조건이다. 아래는 upstream `doc/setup.md` 의 절차다.

> ⚠️ **클론 위치를 이 리포(S15P11A108) 안으로 잡지 말 것.** 체크포인트가 수 GB다.
> 아래는 `~/workspace/sam-3d-objects` 기준이며, 다른 곳에 뒀으면 §1 의 `SAM3D_REPO`
> 상수(또는 환경변수 `SAM3D_REPO`)만 바꾸면 된다.

```bash
cd ~/workspace
git clone https://github.com/facebookresearch/sam-3d-objects.git
cd sam-3d-objects

# 전용 환경 — qual/ai_env 등 기존 환경에 섞지 말 것 (torch 2.5.1+cu121 로 핀이 박혀 있다)
mamba env create -f environments/default.yml   # conda 로 대체 가능
mamba activate sam3d-objects

export PIP_EXTRA_INDEX_URL="https://pypi.ngc.nvidia.com https://download.pytorch.org/whl/cu121"
pip install -e '.[dev]'
pip install -e '.[p3d]'
export PIP_FIND_LINKS="https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.1_cu121.html"
pip install -e '.[inference]'
./patching/hydra

# gated 체크포인트 — 모델 페이지에서 접근 승인을 받은 계정으로 로그인해야 받아진다
pip install 'huggingface-hub[cli]<1.0'
hf auth login
TAG=hf
hf download --repo-type model --local-dir checkpoints/${TAG}-download \
  --max-workers 1 facebook/sam-3d-objects
mv checkpoints/${TAG}-download/checkpoints checkpoints/${TAG}
rm -rf checkpoints/${TAG}-download

# 이 노트북을 그 환경 커널로 띄우기 위해
pip install ipykernel
python -m ipykernel install --user --name sam3d-objects
```

그리고 **이 노트북의 커널을 `sam3d-objects` 로 바꿔서** 실행한다.
추가 의존성은 없다 — `imageio`/`numpy`/`Pillow` 는 위 환경에 이미 들어 있다.

> `hf` CLI 가 없다는 오류가 나면 구버전이다 — `huggingface-cli download ...` 로 바꿔 쓴다.

## 1. 설정 상수 + 경로 등록

`validation/model_selection/model_registry.py` 관례대로 **바꿔 끼울 값을 한곳에 모은다.**
입력 이미지·마스크·렌더 파라미터는 전부 여기서만 고치면 된다.

In [ ]:
import os
import sys
import time
from pathlib import Path

# ── SAM 3D Objects 리포 위치 ────────────────────────────────────────────────
# 이 안의 notebook/inference.py 를 임포트해서 쓴다(설치되는 패키지에 없는 래퍼다).
SAM3D_REPO = Path(os.environ.get("SAM3D_REPO", Path.home() / "workspace" / "sam-3d-objects"))
CHECKPOINT_TAG = "hf"  # setup.md 의 TAG=hf — facebook/sam-3d-objects 다운로드본

# ── 입력 ────────────────────────────────────────────────────────────────────
# 트라이온 스파이크(AI_tryon/FLUX) 산출물을 기본 입력으로 쓴다.
TRYON_IMAGE = Path("../AI_tryon/FLUX/tryon_output.png").resolve()

# 피사체 마스크 PNG. None 이면 §2 에서 배경 flood fill 로 자동 생성한다.
# 마스크 품질이 결과를 크게 좌우하므로, 자동 마스크가 부실하면 qual 환경의
# sam3-condition-check 노트북(SAM3 PCS, prompt="a person")으로 뽑아 저장한 PNG 를 넣는다.
MASK_PATH = None
BG_THRESH = 60  # 자동 마스크의 배경색 허용 오차(RGB 절대차의 합, PIL floodfill thresh)

# ── 출력 ────────────────────────────────────────────────────────────────────
# 절대경로로 고정한다 — 렌더/저장 경로가 cwd 에 흔들리지 않게.
OUTPUT_DIR = Path("outputs").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# ── 360° 턴테이블 렌더 파라미터 (upstream demo_single_object.ipynb 와 동일값) ──
# VRAM 이 모자라면 resolution 을 먼저 낮춘다(384 -> 256).
RENDER_KWARGS = {"r": 1, "fov": 60, "pitch_deg": 15, "yaw_start_deg": -45, "resolution": 512}
GIF_FPS = 30

# ── §6 크롭 비교용 ──────────────────────────────────────────────────────────
# None 이면 마스크 bbox 를 쓴다. 상의만 3D 로 뽑아 보려면 [x1, y1, x2, y2] 를 직접 넣는다.
GARMENT_BBOX = None
CROP_PAD_RATIO = 0.06

# ── 경로 등록 ───────────────────────────────────────────────────────────────
_notebook_dir = SAM3D_REPO / "notebook"
if not _notebook_dir.is_dir():
    print(f"[에러] SAM3D_REPO 아래에 notebook/ 이 없습니다: {SAM3D_REPO}")
    print("  §0 의 클론 절차를 먼저 수행하거나, SAM3D_REPO 상수를 실제 클론 경로로 고치세요.")
    raise FileNotFoundError(_notebook_dir)

if str(_notebook_dir) not in sys.path:
    sys.path.insert(0, str(_notebook_dir))

print(f"SAM3D_REPO  : {SAM3D_REPO}")
print(f"입력 이미지 : {TRYON_IMAGE}  (존재: {TRYON_IMAGE.is_file()})")
print(f"출력 폴더   : {OUTPUT_DIR}")

## 1-1. CUDA / VRAM 확인

**이 노트북의 1번 질문이 여기서 시작된다.** upstream README 는 VRAM 요구치를 명시하지 않았고,
SAM 3D Objects 는 flow-matching 기반 생성 모델이라 8GB 는 낙관적이지 않다.
총량을 먼저 찍어 두고 §1-2·§3 의 실측치와 비교해서 읽는다.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"torch          : {torch.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")

if DEVICE == "cuda":
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu            : {_props.name}")
    print(f"total VRAM     : {_props.total_memory / 1024**3:.2f} GB")
    print(f"cuda runtime   : {torch.version.cuda}")
else:
    print("[주의] CUDA 를 찾지 못했습니다. 이 모델의 CPU 추론은 실용적이지 않습니다.")


def vram_report(tag):
    """(allocated_MB, peak_MB) 를 찍고 반환."""
    if DEVICE != "cuda":
        print(f"{tag:<26} (CPU 모드 — VRAM 측정 없음)")
        return (0.0, 0.0)
    alloc = torch.cuda.memory_allocated() / 1024**2
    peak = torch.cuda.max_memory_allocated() / 1024**2
    print(f"{tag:<26} allocated={alloc:9.1f} MB ({alloc / 1024:.2f} GB)   peak={peak:9.1f} MB")
    return (alloc, peak)

## 1-2. `inference.py` 임포트 + 모델 로드

upstream `notebook/demo_single_object.ipynb` 와 같은 진입점을 쓴다
(`Inference`, `load_image`, `display_image`, `make_scene`,
`ready_gaussian_for_video_rendering`, `render_video`).

`interactive_visualizer` 는 버전에 따라 없을 수도 있어서 따로 감싸 임포트한다 —
없으면 §5 만 건너뛰고 나머지는 그대로 돈다.

In [ ]:
try:
    from inference import (
        Inference,
        display_image,
        load_image,
        make_scene,
        ready_gaussian_for_video_rendering,
        render_video,
    )
except Exception as exc:
    print(f"[에러] inference 모듈 임포트 실패: {type(exc).__name__}: {exc}")
    print("  1) 커널이 sam3d-objects 환경인지 확인 (§0 의 ipykernel 등록)")
    print("  2) pip install -e '.[dev]' / '.[p3d]' / '.[inference]' + ./patching/hydra 완료 여부")
    print(f"  3) sys.path 에 들어간 경로: {_notebook_dir}")
    raise

try:
    from inference import interactive_visualizer
except ImportError:
    interactive_visualizer = None
    print("[정보] interactive_visualizer 가 없는 버전입니다 — §5 는 건너뜁니다.")

In [ ]:
CONFIG_PATH = SAM3D_REPO / "checkpoints" / CHECKPOINT_TAG / "pipeline.yaml"

if not CONFIG_PATH.is_file():
    print(f"[에러] 체크포인트 설정을 찾지 못했습니다: {CONFIG_PATH}")
    print("  §0 의 `hf download ... facebook/sam-3d-objects` 절차가 끝나야 생깁니다.")
    print("  403 이 났다면 토큰이 아니라 gated 접근 승인 문제입니다 — 모델 페이지에서 확인.")
    raise FileNotFoundError(CONFIG_PATH)

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
vram_report("로드 전")

_t0 = time.perf_counter()
inference = Inference(str(CONFIG_PATH), compile=False)
print(f"모델 로드 {time.perf_counter() - _t0:.1f}s")

load_alloc, load_peak = vram_report("로드 후")

## 2. 트라이온 이미지 + 피사체 마스크

SAM 3D Objects 는 `(이미지, 마스크)` 쌍을 받는다 — 마스크가 "무엇을 복원할지"를 정하므로
**마스크 품질이 결과 품질을 그대로 좌우한다.**

트라이온 출력은 백색 스튜디오 배경이라 **테두리에서 flood fill** 하면 피사체 마스크가 대충
나온다. 별도 모델이 필요 없어 이 스파이크에는 충분하지만, 정식 검증에는 SAM3 PCS 마스크를
쓰는 게 맞다(`MASK_PATH` 상수로 갈아끼운다).

> flood fill 은 **네 모서리 + 상단 중앙**을 시드로 쓴다. 피사체가 이미지 테두리에 닿아 있으면
> 그 부분이 배경으로 먹혀 잘려 나간다 — 아래 시각화에서 그것부터 확인한다.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw


def auto_subject_mask(pil_image, thresh=BG_THRESH, sentinel=(255, 0, 255)):
    """테두리에서 flood fill 해 '배경이 아닌 영역'을 피사체 마스크로 삼는다.

    PIL 의 floodfill 은 후보 픽셀을 **시드 픽셀의 원래 색**과 비교하므로(이웃과 비교하지
    않는다), 배경에 그라데이션이 있으면 중간에서 멈춘다. 그래서 시드를 여러 곳에 둔다.

    Returns:
        np.ndarray(bool): (H, W) — True 가 피사체.
    """
    canvas = pil_image.convert("RGB").copy()
    width, height = canvas.size
    seeds = [
        (0, 0),
        (width - 1, 0),
        (0, height - 1),
        (width - 1, height - 1),
        (width // 2, 0),
    ]
    for seed in seeds:
        ImageDraw.floodfill(canvas, seed, sentinel, thresh=thresh)

    arr = np.array(canvas)
    filled = np.all(arr == np.array(sentinel, dtype=np.uint8), axis=-1)
    return ~filled


image = load_image(TRYON_IMAGE)
pil_image = Image.open(TRYON_IMAGE).convert("RGB")

if MASK_PATH is not None:
    mask = np.array(Image.open(MASK_PATH).convert("L")) > 127
    print(f"마스크: 파일 {MASK_PATH}")
else:
    mask = auto_subject_mask(pil_image)
    print(f"마스크: 자동 생성 (flood fill, thresh={BG_THRESH})")

if mask.shape != image.shape[:2]:
    raise ValueError(f"마스크 크기 불일치: mask={mask.shape} image={image.shape[:2]}")

print(f"이미지     : {image.shape} {image.dtype}")
print(f"피사체 비율: {mask.mean() * 100:.1f}%  (전신 스튜디오 샷이면 대략 10~35% 가 정상)")

display_image(image, masks=[mask])

## 3. 3D 재구성 실행 — 질문 1·2 의 답

`inference(image, mask, seed=SEED)` 한 번. **소요 시간**과 **VRAM 피크**를 함께 찍는다.

`torch.OutOfMemoryError` 가 나면 그게 곧 결과다 — 8GB 로는 안 된다는 뜻이고, 그때는
더 큰 VRAM 머신(팀 공용 GPU)에서 재시도해야 한다. 여기서 억지로 우회하지 않는다.

In [ ]:
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

_t0 = time.perf_counter()
output = inference(image, mask, seed=SEED)
recon_sec = time.perf_counter() - _t0

print(f"재구성 소요 : {recon_sec:.1f}s")
_, recon_peak = vram_report("재구성 후")
print(f"출력 키     : {sorted(output.keys())}")

ply_path = OUTPUT_DIR / f"{TRYON_IMAGE.stem}.ply"
output["gs"].save_ply(str(ply_path))
print(f"splat 저장  : {ply_path}  ({ply_path.stat().st_size / 1024**2:.1f} MB)")

## 4. 360° 턴테이블 렌더 → GIF → 화면 표시

여기까지 오면 "화면에 3D 로 보여주기"가 끝난다. `make_scene` 으로 splat 을 씬에 올리고
`render_video` 로 한 바퀴 돌린 프레임을 GIF 로 묶어 인라인 표시한다.

GIF 는 파일 경로 대신 **바이트를 그대로 임베드**해서 띄운다(cwd 나 브라우저 캐시에 흔들리지
않게). 파일도 `outputs/` 에 남으니 슬랙·문서에 그대로 붙일 수 있다.

**볼 것**: 정면은 원래 이미지가 있으니 당연히 그럴듯하다. **측면·후면이 무너지는지**가 판단
포인트다 — 단일 이미지 재구성은 안 보이는 면을 생성으로 채운다.

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image as ImageDisplay


def render_turntable_gif(model_output, gif_path, render_kwargs=None):
    """splat 을 한 바퀴 돌려 GIF 로 저장하고 (경로, 프레임수) 를 반환."""
    scene = make_scene(model_output)
    scene = ready_gaussian_for_video_rendering(scene)
    frames = render_video(scene, **(render_kwargs or RENDER_KWARGS))["color"]
    imageio.mimsave(gif_path, frames, format="GIF", duration=1000 / GIF_FPS, loop=0)
    return gif_path, len(frames)


gif_path = OUTPUT_DIR / f"{TRYON_IMAGE.stem}_turntable.gif"

_t0 = time.perf_counter()
_, n_frames = render_turntable_gif(output, gif_path)
print(f"렌더 {time.perf_counter() - _t0:.1f}s / {n_frames} frames -> {gif_path}")
vram_report("렌더 후")

ImageDisplay(data=gif_path.read_bytes(), format="gif")

## 5. (선택) 인터랙티브 뷰어

`interactive_visualizer(ply_path)` 는 노트북 안에 마우스로 돌릴 수 있는 뷰어를 띄운다.
GIF 로 판단이 안 될 때만 쓴다 — splat 전체를 브라우저로 넘기므로 무겁다.

`.ply` 파일 자체는 [SuperSplat](https://superspl.at/editor) 같은 외부 뷰어로도 열린다.
FE 팀에 "이런 걸 붙일 수 있다"를 보여줄 때는 그쪽이 편하다.

In [ ]:
if interactive_visualizer is None:
    print("interactive_visualizer 를 임포트하지 못했습니다 — 건너뜁니다.")
else:
    interactive_visualizer(str(ply_path))

## 6. (선택) 전신 통짜 vs 크롭 비교 — 질문 5

논문 F절은 **단일 객체를 크롭하면 상대 해상도가 올라가 복원이 훨씬 잘 된다**고 명시한다.
"옷 입은 전신"을 통짜로 넣는 §3 과, 크롭해서 넣은 결과를 나란히 놓고 그 주장을 확인한다.

- `GARMENT_BBOX = None` (기본) → **마스크 bbox 로 크롭.** 배경 여백만 제거하는 셈이라
  "여백 제거만으로도 좋아지는가"를 본다.
- `GARMENT_BBOX = [x1, y1, x2, y2]` → **의류 영역만 크롭.** 상품 의류 3D 카탈로그 방향의
  가능성을 보는 쪽이다. §2 의 `display_image` 그림에서 좌표를 읽어 직접 넣는다.

> 크롭 마스크는 원본 마스크를 같이 자른 것이다. 의류 bbox 로 자르면 마스크에 팔·목이 남아
> 그 부분도 함께 복원된다 — **의류만 남긴 마스크가 아니면 "의류 단독 에셋"이 되지는 않는다.**
> 이 셀은 어디까지 되는지 감을 잡는 용도다.

In [ ]:
RUN_CROP_COMPARISON = True  # 무거우면(§3 가 느렸다면) False 로 두고 넘어간다


def bbox_from_mask(bool_mask, pad_ratio=CROP_PAD_RATIO):
    """마스크의 tight bbox 를 pad_ratio 만큼 넓혀 [x1, y1, x2, y2] 로 반환."""
    ys, xs = np.nonzero(bool_mask)
    if len(xs) == 0:
        raise ValueError("마스크가 비어 있습니다 — BG_THRESH 를 조정하세요.")
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    pad_x = int((x2 - x1) * pad_ratio)
    pad_y = int((y2 - y1) * pad_ratio)
    height, width = bool_mask.shape
    return [
        max(0, x1 - pad_x),
        max(0, y1 - pad_y),
        min(width, x2 + pad_x),
        min(height, y2 + pad_y),
    ]


def crop_pair(img_arr, bool_mask, bbox):
    """이미지와 마스크를 같은 bbox 로 자른다."""
    x1, y1, x2, y2 = bbox
    return img_arr[y1:y2, x1:x2], bool_mask[y1:y2, x1:x2]


if RUN_CROP_COMPARISON:
    crop_bbox = GARMENT_BBOX or bbox_from_mask(mask)
    crop_image, crop_mask = crop_pair(image, mask, crop_bbox)
    _ch, _cw = crop_mask.shape
    _fh, _fw = mask.shape
    print(f"bbox        : {crop_bbox}  ({'수동 GARMENT_BBOX' if GARMENT_BBOX else '마스크 bbox'})")
    print(f"크기 변화   : {_fw}x{_fh} -> {_cw}x{_ch}")
    print(f"피사체 비율 : {mask.mean() * 100:.1f}% -> {crop_mask.mean() * 100:.1f}%")
    display_image(crop_image, masks=[crop_mask])

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    _t0 = time.perf_counter()
    crop_output = inference(crop_image, crop_mask, seed=SEED)
    print(f"재구성 소요 : {time.perf_counter() - _t0:.1f}s")
    vram_report("크롭 재구성 후")

    crop_ply = OUTPUT_DIR / f"{TRYON_IMAGE.stem}_crop.ply"
    crop_output["gs"].save_ply(str(crop_ply))

    crop_gif = OUTPUT_DIR / f"{TRYON_IMAGE.stem}_crop_turntable.gif"
    render_turntable_gif(crop_output, crop_gif)
    print(f"저장        : {crop_ply.name}, {crop_gif.name}")
else:
    crop_gif = None
    print("RUN_CROP_COMPARISON=False — 건너뜁니다.")

### 비교 표시 — 위: 전신 통짜 / 아래: 크롭

같은 화면에서 위아래로 놓고 본다. **판단 기준**은 옷의 주름·프린트·실루엣이 크롭 쪽에서
얼마나 더 살아 있는지다. 차이가 미미하면 "전신 통짜로 충분", 크게 좋아지면 3D 활용은
**의류 단독 크롭 방향**으로 좁힌다.

In [ ]:
from IPython.display import display

print("── 전신 통짜 (§3) ──")
display(ImageDisplay(data=gif_path.read_bytes(), format="gif"))

if crop_gif is not None:
    print("── 크롭 (§6) ──")
    display(ImageDisplay(data=crop_gif.read_bytes(), format="gif"))

---

## 결론 (실행 후 채울 것)

| # | 확인 항목 | 근거 셀 | 결과 |
| --- | --- | --- | --- |
| 1 | 8GB VRAM 에서 재구성이 완주하는가 (피크 VRAM 몇 GB) | §1-2, §3 | |
| 2 | 1장당 소요 시간 (모델 로드 / 재구성 / 렌더) | §1-2, §3, §4 | |
| 3 | flood fill 자동 마스크가 쓸 만한가, SAM3 마스크가 필요한가 | §2 | |
| 4 | 측면·후면 품질이 "360° 뷰어"로 내보낼 수준인가 | §4 | |
| 5 | 크롭이 실제로 품질을 올리는가 (논문 F절 주장 검증) | §6 | |

### 이 결과로 갈리는 판단

- **1이 실패**하면 3D 논의는 여기서 중단한다. 8GB 로 안 되면 팀 노트북 환경에서 서비스 경로에
  못 올린다(`AI/docs/infra-architecture.md` §3.2 — 추론은 팀원 노트북 워커에서 돈다).
- **4가 괜찮고 5의 차이가 작으면** → 정적 360° 뷰어(부가 기능) 방향.
- **5의 차이가 크면** → 상품 의류 3D 카탈로그 방향. 그때는 "의류만 남긴 마스크"가 필요하므로
  SAM3 PCS(`prompt="a t-shirt"` 등)로 의류 마스크를 먼저 뽑는 단계가 추가된다.

### 파인튜닝(AI Hub 데이터)은 이 노트북 다음 단계다

성능이 확인된 뒤에 판단할 항목이며, 최소 두 조건이 먼저 확인돼야 한다 —
**(a) 실사 이미지와 그 사진의 3D 메시가 페어링돼 있는가, (b) 샘플별 카메라 extrinsic 이
있는가.** 둘 중 하나라도 없으면 논문이 의도한 visually-grounded SFT 가 아니라 합성 렌더
pretraining 보강에 가까워지고, in-the-wild 개선 효과가 급감한다. 이 노트북은 그 판단의
전제(모델이 우리 환경에서 도는가)만 확인한다.